# Assignment 2 — Question 5: Enhancing K-Means with Association Rule Mining

Goal: improve K-Means clustering of `mobile_price.csv` (4 classes) by using FP-growth association rule mining to (i) re-weight features in the distance metric and (ii) seed cluster centroids.

**Inspirations:**
- Q3 finding: in `price_range==1`, `ram_medium` has support 0.682 — RAM is by far the most concentrated feature in the segment.
- Q4 finding: vanilla K-Means on standardised features achieves ARI ≈ 0.006 (essentially zero). Diagnosis: Euclidean distance treats all 20 features as equally informative, but the price signal is concentrated in a small subset.
- Lecture 8: k-means++ shows that *initialisation strategy* meaningfully affects final clustering — we extend the same idea to ARM-derived prototypes.
- Jinbo Shang, DSC148 HW2/HW3 (assignment v2 reference).
- *Theoretical context:* the proposed weighted-Euclidean distance is a special case of **Mahalanobis distance** with a diagonal precision matrix derived from ARM lift scores. We implement it as plain feature rescaling (multiplying feature j by √wⱼ) for efficiency — this is mathematically identical to using Mahalanobis distance with `M = diag(w)` inside K-Means, but reuses sklearn's standard implementation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics.cluster import adjusted_rand_score
from scipy.optimize import linear_sum_assignment

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

os.makedirs('chart', exist_ok=True)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)

SEEDS = [0, 10, 42, 100, 999]
N_CLUSTERS = 4

## 1. Load data and identify continuous vs binary features

In [ ]:
df = pd.read_csv('data/mobile_price.csv')
y = df['price_range'].values
X_df = df.drop(columns=['price_range'])

# Identify binary vs continuous features (binary = exactly 2 unique values)
binary_feats     = [c for c in X_df.columns if X_df[c].nunique() == 2]
continuous_feats = [c for c in X_df.columns if X_df[c].nunique() > 2]

print(f'Total features:      {X_df.shape[1]}')
print(f'Binary features ({len(binary_feats)}):    {binary_feats}')
print(f'Continuous features ({len(continuous_feats)}): {continuous_feats}')

# Standardised version (for K-Means)
scaler = StandardScaler()
X_std  = scaler.fit_transform(X_df.values.astype(float))
FEATURES = list(X_df.columns)

## 2. Discretise features for ARM

- Continuous features → `low / medium / high` via the same 3 : 4 : 3 ratio as Q3.
- Binary features stay as `feat_0` / `feat_1`.
- The class label is appended as item `price_X` so we can mine **class-conditional rules**: `{feature buckets} → {price_X}`.

In [ ]:
def discretise_3_4_3(series, name):
    lo, hi = series.min(), series.max()
    rng = hi - lo
    bins = [-np.inf, lo + 0.3*rng, lo + 0.7*rng, np.inf]
    labels = [f'{name}_low', f'{name}_medium', f'{name}_high']
    return pd.cut(series, bins=bins, labels=labels, right=False).astype(str)

# Build per-row item lists
transactions = []
for idx in df.index:
    items = []
    for f in continuous_feats:
        items.append(discretise_3_4_3(X_df[f], f).iloc[idx])
    for f in binary_feats:
        items.append(f'{f}_{int(X_df[f].iloc[idx])}')
    items.append(f'price_{int(y[idx])}')
    transactions.append(items)

te = TransactionEncoder()
trans_arr = te.fit(transactions).transform(transactions)
trans_df  = pd.DataFrame(trans_arr, columns=te.columns_)

print(f'Transactions:    {len(transactions)}')
print(f'Distinct items:  {trans_df.shape[1]}')
print('First transaction:')
print(' ', transactions[0])

## 3. Diagnostic ARM — class-conditional rules across **all four** price ranges

Q3 only inspected `price_range==1`. Here we mine rules `{feature buckets} → price_X` for every X, then derive a **per-feature discriminative score** = max lift across all rules whose antecedent contains any bucket of that feature. This score will become the K-Means feature weight.

Low support threshold (0.05) is used because we are mining within each class; class-specific patterns may not exceed 30% of the *whole* dataset.

In [ ]:
MIN_SUPPORT = 0.05
MIN_LIFT    = 1.0   # we want positive associations
MIN_CONF    = 0.4

freq = fpgrowth(trans_df, min_support=MIN_SUPPORT, use_colnames=True)
all_rules = association_rules(freq, metric='lift', min_threshold=MIN_LIFT)
all_rules = all_rules[all_rules['confidence'] >= MIN_CONF]

# Keep only rules whose consequent is a single class label
def is_class_rule(consequents):
    return len(consequents) == 1 and list(consequents)[0].startswith('price_')

class_rules = all_rules[all_rules['consequents'].apply(is_class_rule)].copy()
class_rules['target_class'] = class_rules['consequents'].apply(lambda s: list(s)[0])
class_rules['antecedents_str'] = class_rules['antecedents'].apply(lambda s: ', '.join(sorted(s)))
class_rules = class_rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Class-conditional rules found: {len(class_rules)}')
print('\nTop 10 by lift:')
print(class_rules[['target_class', 'antecedents_str', 'support', 'confidence', 'lift']]
      .head(10).round(3).to_string(index=False))

In [ ]:
def per_feature_max_lift(rules_df, features):
    """Per-feature discriminative score = max lift across rules whose antecedent contains any bucket of this feature."""
    scores = {}
    for f in features:
        # Match: antecedent item starts with f + '_'
        mask = rules_df['antecedents'].apply(
            lambda s: any(item.startswith(f + '_') for item in s)
        )
        if mask.sum() == 0:
            scores[f] = 1.0  # baseline weight (no rule found)
        else:
            scores[f] = float(rules_df.loc[mask, 'lift'].max())
    return scores

feat_scores = per_feature_max_lift(class_rules, FEATURES)
scores_df = pd.DataFrame.from_dict(feat_scores, orient='index', columns=['max_lift'])
scores_df['rank'] = scores_df['max_lift'].rank(ascending=False).astype(int)
scores_df = scores_df.sort_values('max_lift', ascending=False)
print('=== Per-feature discriminative score (max lift across class-conditional rules) ===')
print(scores_df.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
scores_sorted = scores_df.sort_values('max_lift', ascending=True)
colors = ['#1E88E5' if v <= 1.0 else '#43A047' if v <= 1.5 else '#FB8C00' if v <= 2.0 else '#E53935'
          for v in scores_sorted['max_lift']]
ax.barh(scores_sorted.index, scores_sorted['max_lift'], color=colors, edgecolor='black')
ax.axvline(1.0, color='black', linestyle='--', alpha=0.4, label='lift = 1 (independent)')
ax.set_xlabel('Max lift across class-conditional rules')
ax.set_title('Per-feature discriminative score (input to K-Means weighting)')
ax.legend()
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig('chart/Q5_feature_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Helper functions — cluster→label assignment and metric evaluation

Cluster IDs from K-Means are arbitrary, so we need a mapping cluster→true-class to compute classification metrics.

Two strategies:
1. **Majority vote** (course-standard): each cluster takes the label that appears most often within it.
2. **Hungarian alignment** (optimal): solve `linear_sum_assignment` on the negative confusion matrix to find the one-to-one mapping that maximises overall accuracy.

We compute metrics under both and verify they agree.

In [ ]:
def majority_vote_map(cluster_labels, y_true, n_clusters=N_CLUSTERS):
    mapping = {}
    for k in range(n_clusters):
        mask = cluster_labels == k
        if mask.sum() == 0:
            mapping[k] = 0
        else:
            mapping[k] = int(pd.Series(y_true[mask]).mode().iloc[0])
    return np.array([mapping[k] for k in cluster_labels])

def hungarian_map(cluster_labels, y_true, n_clusters=N_CLUSTERS):
    # Build confusion matrix [cluster x class]
    cm = np.zeros((n_clusters, n_clusters), dtype=int)
    for k in range(n_clusters):
        for c in range(n_clusters):
            cm[k, c] = ((cluster_labels == k) & (y_true == c)).sum()
    # Maximise matching = minimise negative
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {int(r): int(c) for r, c in zip(row_ind, col_ind)}
    return np.array([mapping[k] for k in cluster_labels])

def evaluate(cluster_labels, y_true, alignment='majority'):
    if alignment == 'majority':
        y_pred = majority_vote_map(cluster_labels, y_true)
    elif alignment == 'hungarian':
        y_pred = hungarian_map(cluster_labels, y_true)
    else:
        raise ValueError(alignment)
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1':        f1_score(y_true, y_pred, average='macro', zero_division=0),
        'ari':       adjusted_rand_score(y_true, cluster_labels),
        'y_pred':    y_pred,
    }

## 5. Method definitions — baseline and the three ARM enhancements

**M1 Baseline:** vanilla K-Means on standardised features, default `k-means++` init.

**M2 ARM-Init:** Initialise centroids as the mean of training samples that match the strongest class-conditional rule per class. Falls back to class mean if no rule covers a class.

**M3 ARM-Weighted:** Re-scale each feature by √wⱼ where wⱼ = max-lift score of feature j (computed in section 3), then standard K-Means with `k-means++` init. Mathematically equivalent to Mahalanobis distance with a diagonal precision matrix.

**M4 Full = ARM-Init + ARM-Weighted:** combine both.

In [ ]:
def get_arm_centroids(X_input, y_true, class_rules_df, n_clusters=N_CLUSTERS):
    """For each class, take the centroid of samples matching its top class-conditional rule.
    Falls back to class mean if no rule available."""
    centroids = []
    for c in range(n_clusters):
        target = f'price_{c}'
        sub = class_rules_df[class_rules_df['target_class'] == target]
        if len(sub) == 0:
            centroids.append(X_input[y_true == c].mean(axis=0))
            continue
        # Top rule by lift
        top = sub.sort_values('lift', ascending=False).iloc[0]
        antecedent = top['antecedents']
        # Find rows in trans_df matching all antecedent items
        match = np.ones(len(trans_df), dtype=bool)
        for item in antecedent:
            match &= trans_df[item].values
        if match.sum() == 0:
            centroids.append(X_input[y_true == c].mean(axis=0))
        else:
            centroids.append(X_input[match].mean(axis=0))
    return np.array(centroids)

def apply_weights(X_input, weights_dict, features):
    w = np.array([weights_dict[f] for f in features])
    return X_input * np.sqrt(w)[None, :]

# Pre-compute the inputs that don't depend on seed
X_weighted = apply_weights(X_std, feat_scores, FEATURES)
init_centroids_std      = get_arm_centroids(X_std, y, class_rules)
init_centroids_weighted = get_arm_centroids(X_weighted, y, class_rules)

print(f'Weighted feature space shape: {X_weighted.shape}')
print(f'ARM init centroids (std space) shape: {init_centroids_std.shape}')

## 6. Run all 4 methods × 5 seeds × 2 alignment strategies

In [ ]:
def run_kmeans(X_in, init, seed):
    """init: 'k-means++' string OR ndarray of centroids."""
    if isinstance(init, str):
        km = KMeans(n_clusters=N_CLUSTERS, init=init, n_init=10, random_state=seed)
    else:
        km = KMeans(n_clusters=N_CLUSTERS, init=init, n_init=1, random_state=seed)
    return km.fit_predict(X_in)

methods = {
    'M1_Baseline':       (X_std,      'k-means++'),
    'M2_ARM_Init':       (X_std,      init_centroids_std),
    'M3_ARM_Weighted':   (X_weighted, 'k-means++'),
    'M4_Full':           (X_weighted, init_centroids_weighted),
}

all_records = []
raw_labels = {}
for mname, (X_in, init) in methods.items():
    for seed in SEEDS:
        labs = run_kmeans(X_in, init, seed)
        raw_labels[(mname, seed)] = labs
        for align in ['majority', 'hungarian']:
            r = evaluate(labs, y, alignment=align)
            all_records.append({
                'method': mname, 'seed': seed, 'alignment': align,
                'accuracy': r['accuracy'], 'precision': r['precision'],
                'recall':   r['recall'],   'f1':        r['f1'],
                'ari':      r['ari'],
            })

all_df = pd.DataFrame(all_records)
print(f'Total rows: {len(all_df)}  (4 methods × 5 seeds × 2 alignments = 40)')
all_df.head(8)

### 6.1 Per-seed table

In [ ]:
for align in ['majority', 'hungarian']:
    print(f'\n=== Alignment: {align} ===')
    sub = all_df[all_df['alignment'] == align]
    pivot = sub.pivot_table(index='method', columns='seed', values='accuracy').round(4)
    print('\nAccuracy per seed:')
    print(pivot.to_string())
    pivot_f1 = sub.pivot_table(index='method', columns='seed', values='f1').round(4)
    print('\nMacro F1 per seed:')
    print(pivot_f1.to_string())

### 6.2 Mean ± std summary

In [ ]:
for align in ['majority', 'hungarian']:
    print(f'\n=== Alignment: {align} ===')
    sub = all_df[all_df['alignment'] == align]
    summary = sub.groupby('method').agg(
        acc_mean=('accuracy','mean'), acc_std=('accuracy','std'),
        prec_mean=('precision','mean'), prec_std=('precision','std'),
        rec_mean=('recall','mean'),   rec_std=('recall','std'),
        f1_mean=('f1','mean'),        f1_std=('f1','std'),
        ari_mean=('ari','mean'),      ari_std=('ari','std'),
    ).round(4)
    print(summary.to_string())

## 7. Per-class confusion matrices — Baseline (M1) vs Full (M4)

Use `seed=42` for reproducibility, majority-vote alignment. Show side-by-side.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

labs_m1 = raw_labels[('M1_Baseline', 42)]
labs_m4 = raw_labels[('M4_Full',     42)]

y_pred_m1 = majority_vote_map(labs_m1, y)
y_pred_m4 = majority_vote_map(labs_m4, y)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_pred, title in zip(
    axes, [y_pred_m1, y_pred_m4],
    [f'M1 Baseline KMeans  (acc={accuracy_score(y, y_pred_m1):.3f})',
     f'M4 Full ARM-KMeans  (acc={accuracy_score(y, y_pred_m4):.3f})']
):
    cm = confusion_matrix(y, y_pred, labels=[0,1,2,3])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(['0','1','2','3'])
    ax.set_yticklabels(['0','1','2','3'])
    ax.set_xlabel('Predicted price_range')
    ax.set_ylabel('True price_range')
    ax.set_title(title)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
fig.savefig('chart/Q5_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== M1 Baseline classification report ===')
print(classification_report(y, y_pred_m1, digits=3, zero_division=0))
print('=== M4 Full classification report ===')
print(classification_report(y, y_pred_m4, digits=3, zero_division=0))

## 8. Sensitivity analysis — vary the support threshold used to mine rules

The proposed method has two thresholds: `min_support` (controls which itemsets become rules) and `min_lift` (controls which rules are kept). Vary `min_support` over a reasonable range and re-run M3 (ARM-Weighted) to confirm the method isn't sensitive to one specific cherry-picked threshold.

In [ ]:
supports = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30]
sens_records = []

for ms in supports:
    fp_s = fpgrowth(trans_df, min_support=ms, use_colnames=True)
    if len(fp_s) == 0:
        continue
    rules_s = association_rules(fp_s, metric='lift', min_threshold=MIN_LIFT)
    rules_s = rules_s[rules_s['confidence'] >= MIN_CONF]
    rules_s = rules_s[rules_s['consequents'].apply(is_class_rule)].copy()
    if len(rules_s) == 0:
        continue
    weights_s = per_feature_max_lift(rules_s, FEATURES)
    Xw_s = apply_weights(X_std, weights_s, FEATURES)
    for seed in SEEDS:
        labs = run_kmeans(Xw_s, 'k-means++', seed)
        r = evaluate(labs, y, alignment='majority')
        sens_records.append({
            'min_support': ms, 'seed': seed,
            'accuracy': r['accuracy'], 'f1': r['f1'], 'ari': r['ari'],
            'n_rules': len(rules_s),
        })

sens_df = pd.DataFrame(sens_records)
sens_summary = sens_df.groupby('min_support').agg(
    n_rules=('n_rules','first'),
    acc_mean=('accuracy','mean'), acc_std=('accuracy','std'),
    f1_mean=('f1','mean'),        f1_std=('f1','std'),
).round(4)
print(sens_summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(sens_summary.index, sens_summary['acc_mean'], yerr=sens_summary['acc_std'],
            fmt='o-', capsize=4, label='Accuracy', color='#1E88E5')
ax.errorbar(sens_summary.index, sens_summary['f1_mean'], yerr=sens_summary['f1_std'],
            fmt='s-', capsize=4, label='Macro F1', color='#E53935')
ax.axhline(0.25, color='gray', linestyle='--', alpha=0.4, label='random baseline (0.25)')
ax.set_xlabel('min_support')
ax.set_ylabel('Score (mean ± std over 5 seeds)')
ax.set_title('Q5 sensitivity: ARM-Weighted KMeans vs `min_support` threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig('chart/Q5_sensitivity_support.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Three-way comparison: vanilla / PCA-then-KMeans / ARM-Weighted

Q4(d) used PCA to reduce to 2-D before clustering — and lost. Compare those three approaches across the same seed set to make the contrast explicit: PCA reduces *variance* dimensionality (the wrong kind), ARM reduces *informativeness* dimensionality (the right kind).

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_std)

threeway_records = []
for label, X_in in [('Vanilla (20-D)', X_std),
                    ('PCA 2-D',         X_pca),
                    ('ARM-Weighted',    X_weighted)]:
    for seed in SEEDS:
        labs = run_kmeans(X_in, 'k-means++', seed)
        r = evaluate(labs, y, alignment='majority')
        threeway_records.append({
            'method': label, 'seed': seed,
            'accuracy': r['accuracy'], 'f1': r['f1'], 'ari': r['ari'],
        })

tw = pd.DataFrame(threeway_records)
tw_summary = tw.groupby('method').agg(
    acc_mean=('accuracy','mean'), acc_std=('accuracy','std'),
    f1_mean=('f1','mean'),        f1_std=('f1','std'),
    ari_mean=('ari','mean'),      ari_std=('ari','std'),
).round(4)
print(tw_summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(tw_summary))
w = 0.35
ax.bar(x - w/2, tw_summary['acc_mean'], w, yerr=tw_summary['acc_std'],
       capsize=4, label='Accuracy', color='#1E88E5')
ax.bar(x + w/2, tw_summary['f1_mean'],  w, yerr=tw_summary['f1_std'],
       capsize=4, label='Macro F1', color='#E53935')
ax.axhline(0.25, color='gray', linestyle='--', alpha=0.4, label='random (0.25)')
ax.set_xticks(x); ax.set_xticklabels(tw_summary.index)
ax.set_ylabel('Score (mean ± std over 5 seeds)')
ax.set_title('Three-way comparison: feature-space choice for K-Means')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig('chart/Q5_threeway_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Framework diagram

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 9)
ax.axis('off')

def box(x, y, w, h, text, fc='#E3F2FD', ec='#1565C0', fontsize=10, weight='normal'):
    p = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                        linewidth=1.5, facecolor=fc, edgecolor=ec)
    ax.add_patch(p)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, weight=weight, wrap=True)

def arrow(x1, y1, x2, y2, color='#1565C0'):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2),
                                  arrowstyle='->', mutation_scale=18,
                                  linewidth=1.5, color=color))

# Top: input
box(5.0, 7.8, 3.0, 0.7, 'mobile_price.csv\n(2000 × 20 + label)', fc='#FFF3E0', ec='#E65100', weight='bold')

# Two parallel branches
box(0.5, 6.2, 4.5, 1.1, 'StandardScaler\n(z-score per feature)', fc='#E3F2FD')
box(8.0, 6.2, 4.5, 1.1, 'Discretise:\ncontinuous → low/med/high (3:4:3)\nbinary → keep as 0/1', fc='#E3F2FD')
arrow(6.0, 7.8, 2.7, 7.3); arrow(7.0, 7.8, 10.3, 7.3)

# Right branch: rules
box(8.0, 4.7, 4.5, 1.1, 'Append price_X label item\n→ FP-growth (min_support=0.05)', fc='#E8F5E9', ec='#2E7D32')
arrow(10.3, 6.2, 10.3, 5.8)
box(8.0, 3.2, 4.5, 1.1, 'Class-conditional rules:\n{features} → price_X', fc='#E8F5E9', ec='#2E7D32')
arrow(10.3, 4.7, 10.3, 4.3)
box(8.0, 1.7, 4.5, 1.1, 'Per-feature score wⱼ\n= max lift across rules', fc='#FFF9C4', ec='#F9A825')
arrow(10.3, 3.2, 10.3, 2.8)
box(8.0, 0.2, 4.5, 1.1, 'ARM-init centroids\n(top rule per class)', fc='#FFF9C4', ec='#F9A825')
arrow(9.0, 3.2, 9.0, 1.3)

# Left branch: weighted features
box(0.5, 4.7, 4.5, 1.1, 'Apply weights:\nx\'ⱼ = √wⱼ · xⱼ', fc='#FCE4EC', ec='#AD1457')
arrow(2.7, 6.2, 2.7, 5.8)
arrow(8.0, 2.3, 5.0, 5.2, color='#AD1457')  # weights → rescale

# Centre: KMeans
box(4.0, 2.8, 5.0, 1.3,
    'K-Means (k=4)\ninit = ARM centroids   |   features = ARM-weighted',
    fc='#F3E5F5', ec='#6A1B9A', fontsize=10, weight='bold')
arrow(2.7, 4.7, 5.0, 4.1)
arrow(8.0, 0.7, 6.5, 2.8, color='#F9A825')

# Bottom: evaluation
box(4.0, 1.0, 5.0, 1.0, 'Cluster→label mapping:\nMajority vote  ‖  Hungarian',
    fc='#E0F2F1', ec='#00695C')
arrow(6.5, 2.8, 6.5, 2.0)
box(4.0, -0.4, 5.0, 1.0, 'Accuracy / Precision / Recall / F1\n(macro, averaged over 5 seeds)',
    fc='#FFEBEE', ec='#B71C1C', weight='bold')
arrow(6.5, 1.0, 6.5, 0.6)

ax.set_title('Q5 Framework — ARM-Enhanced K-Means', fontsize=13, weight='bold', pad=15)
plt.tight_layout()
fig.savefig('chart/Q5_framework.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Observations & discussion

*(Fill in after running all cells; placeholders below are for the report write-up.)*

### Headline numbers
- **Baseline KMeans (M1):** acc = `…`, F1 = `…` (averaged over 5 seeds, majority-vote alignment).
- **Full ARM-KMeans (M4):** acc = `…`, F1 = `…`.
- **Improvement:** Δacc = `…`, ΔF1 = `…`.

### What each component contributes (ablation, section 6.2)
- ARM-Init alone (M2) vs Baseline (M1): isolates the value of *better starting points*.
- ARM-Weighted alone (M3) vs Baseline (M1): isolates the value of *better distance metric*.
- Full (M4) vs the two singles: shows whether the components are complementary or redundant.

### Majority-vote vs Hungarian alignment
Both alignment strategies are reported in section 6.2. The differences indicate how often the greedy majority-vote left a class uncovered (two clusters voting for the same label). When they agree, both are giving the same answer; when Hungarian is higher, baseline majority-vote was sub-optimal because of cluster collisions.

### Sensitivity (section 8)
Method should be stable across `min_support ∈ [0.02, 0.30]` — confirms the headline result is not cherry-picked at one specific threshold.

### Three-way comparison (section 9)
Vanilla < PCA < ARM-Weighted (expected). The narrative: PCA reduces *variance* dimensionality (the wrong kind), while ARM reduces *informativeness* dimensionality (the right kind) by emphasising features that ARM identifies as class-discriminative.

### Per-class performance (section 7)
Likely pattern: improvement concentrated on the **extreme classes (0 and 3)** because their distinguishing features (very-low or very-high RAM, battery, etc.) get the strongest lift in class-conditional rules. The middle classes (1 and 2) remain harder to separate even with ARM weighting because their feature distributions overlap.

### Limitations & honest assessment
1. The ARM mining uses class labels — this is **semi-supervised** rather than pure unsupervised clustering. If labels are unavailable, an alternative is to mine unsupervised frequent itemsets and use itemset frequency as a proxy for feature importance, but this is weaker.
2. The lift→weight mapping is linear (w = max_lift). Alternatives like log-lift or rank-based weights are easy variations but were not explored here for scope reasons.
3. K-Means is a hard-assignment, spherical-cluster method; even with perfect feature weighting it cannot recover non-convex class boundaries. A more flexible base method (e.g., GMM) would benefit from the same ARM enhancement and is a natural extension.